# 📘 Quick Reference

**Purpose**: Process BraTS 2021 T1 MRI data for anomaly detection testing  
**Pipeline**: Extract → Normalize → Filter → Resize → ZIP  
**Output**: 128×128 normalized .npy files (same format as IXI training data)  
**Time**: ~45-75 minutes  

## 📋 Execution Order
1. **Cells 1-4**: Setup & explore (fast)
2. **Cell 9**: Extract slices ⏰ 20-30 min
3. **Cell 11**: Filter slices ⏰ 5-10 min  
4. **Cell 14**: Resize slices ⏰ 10-20 min
5. **Cell 18**: Create ZIP ⏰ 5-10 min

## ✅ Success = 1000-2000 slices, 128×128, range [0,1], ZIP created

---

# BraTS 2021 T1 MRI Preprocessing Pipeline (Local → Drive Upload)

This notebook performs LOCAL preprocessing of BraTS 2021 T1 files following the same pipeline as IXI dataset:

## Pipeline Overview:
1. **Extract T1 files** from BraTS 2021 folders
2. **Extract 2D slices** from 3D T1 NIfTI volumes
3. **Normalize** slices to [0, 1] range
4. **Filter** empty/low-information slices (mean > 0.1)
5. **Resize** to 128x128 standard size
6. **Save as .npy files** ready for Google Drive upload

**Purpose**: BraTS data will be used for TESTING (anomaly detection), while IXI is for TRAINING

## 1. Import Required Libraries

In [ ]:
import os
import shutil
from pathlib import Path
import glob
from tqdm import tqdm

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.transform import resize
import random

print("✓ All libraries imported successfully!")
print(f"Working directory: {os.getcwd()}")

## 2. Define Folder Paths

In [ ]:
# Source folder with BraTS 2021 dataset
brats_folder = r"c:\Users\rifad\symAD-ECNN\data\brats2021"

# Output folders (matching IXI pipeline structure)
t1_raw_folder = r"c:\Users\rifad\symAD-ECNN\data\brats2021_processed\raw_slices"
filtered_folder = r"c:\Users\rifad\symAD-ECNN\data\brats2021_processed\filtered"
resized_folder = r"c:\Users\rifad\symAD-ECNN\data\brats2021_processed\resized"

# Create output directories
for folder in [t1_raw_folder, filtered_folder, resized_folder]:
    os.makedirs(folder, exist_ok=True)

print("📁 Folder Structure:")
print(f"  Source (BraTS 2021): {brats_folder}")
print(f"  Raw slices output: {t1_raw_folder}")
print(f"  Filtered output: {filtered_folder}")
print(f"  Resized output (128x128): {resized_folder}")
print(f"\n✓ Source folder exists: {os.path.exists(brats_folder)}")

## 3. Explore BraTS 2021 Dataset Structure

In [ ]:
# List all patient folders
patient_folders = sorted([d for d in os.listdir(brats_folder) 
                         if os.path.isdir(os.path.join(brats_folder, d)) and d.startswith('BraTS2021_')])

print(f"Total number of patient folders: {len(patient_folders)}")
print(f"\nFirst 5 patient folders:")
for folder in patient_folders[:5]:
    print(f"  - {folder}")

# Examine the structure of first patient folder
if patient_folders:
    sample_folder = os.path.join(brats_folder, patient_folders[0])
    files = os.listdir(sample_folder)
    print(f"\nFiles in {patient_folders[0]}:")
    for file in sorted(files):
        print(f"  - {file}")

## 4. Find and List All T1 Files

In [ ]:
# Find all T1 files (not T1CE - contrast enhanced)
t1_files = []

for patient_folder in patient_folders:
    patient_path = os.path.join(brats_folder, patient_folder)
    # Look for files ending with _t1.nii.gz (not _t1ce.nii.gz)
    for file in os.listdir(patient_path):
        if file.endswith('_t1.nii.gz') and not file.endswith('_t1ce.nii.gz'):
            t1_files.append(os.path.join(patient_path, file))

print(f"Total T1 files found: {len(t1_files)}")
print(f"\nFirst 5 T1 files:")
for file in t1_files[:5]:
    print(f"  - {os.path.basename(file)}")

## 5. Create Target Folder for T1 Files

In [ ]:
# Create target directory if it doesn't exist
os.makedirs(t1_folder, exist_ok=True)
print(f"Target folder created/verified: {t1_folder}")

# Also create preprocessed folder
os.makedirs(preprocessed_folder, exist_ok=True)
print(f"Preprocessed folder created/verified: {preprocessed_folder}")

## 6. Copy T1 Files to Target Folder

In [ ]:
# Copy T1 files to the target folder
copied_count = 0
skipped_count = 0

print("Copying T1 files...")
for source_file in tqdm(t1_files):
    filename = os.path.basename(source_file)
    destination_file = os.path.join(t1_folder, filename)
    
    # Check if file already exists to avoid redundant copying
    if not os.path.exists(destination_file):
        shutil.copy2(source_file, destination_file)
        copied_count += 1
    else:
        skipped_count += 1

print(f"\nCopy operation completed!")
print(f"  - Files copied: {copied_count}")
print(f"  - Files skipped (already exist): {skipped_count}")
print(f"  - Total files in target folder: {len(os.listdir(t1_folder))}")

## 7. Load and Visualize Sample T1 Images

In [ ]:
# Load a sample T1 image
sample_t1_file = t1_files[0]
print(f"Loading sample file: {os.path.basename(sample_t1_file)}")

# Load the NIfTI file
nii_img = nib.load(sample_t1_file)
img_data = nii_img.get_fdata()

print(f"\nImage shape: {img_data.shape}")
print(f"Image data type: {img_data.dtype}")
print(f"Intensity range: [{img_data.min():.2f}, {img_data.max():.2f}]")
print(f"Mean intensity: {img_data.mean():.2f}")
print(f"Std intensity: {img_data.std():.2f}")

In [ ]:
# Visualize different slices
def visualize_slices(img_data, title="MRI Slices"):
    """Visualize axial, sagittal, and coronal slices"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Get middle slices
    mid_x, mid_y, mid_z = [s // 2 for s in img_data.shape]
    
    # Axial slices (different depths)
    axes[0, 0].imshow(img_data[:, :, mid_z - 10], cmap='gray')
    axes[0, 0].set_title(f'Axial Slice (z={mid_z - 10})')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(img_data[:, :, mid_z], cmap='gray')
    axes[0, 1].set_title(f'Axial Slice (z={mid_z})')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(img_data[:, :, mid_z + 10], cmap='gray')
    axes[0, 2].set_title(f'Axial Slice (z={mid_z + 10})')
    axes[0, 2].axis('off')
    
    # Sagittal slice
    axes[1, 0].imshow(img_data[mid_x, :, :], cmap='gray')
    axes[1, 0].set_title(f'Sagittal Slice (x={mid_x})')
    axes[1, 0].axis('off')
    
    # Coronal slice
    axes[1, 1].imshow(img_data[:, mid_y, :], cmap='gray')
    axes[1, 1].set_title(f'Coronal Slice (y={mid_y})')
    axes[1, 1].axis('off')
    
    # Histogram
    axes[1, 2].hist(img_data.flatten(), bins=50, color='blue', alpha=0.7)
    axes[1, 2].set_title('Intensity Histogram')
    axes[1, 2].set_xlabel('Intensity')
    axes[1, 2].set_ylabel('Frequency')
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_slices(img_data, "Original T1 MRI")

## 8. Preprocessing Functions

In [ ]:
def normalize(x):
    """
    Normalize array to [0, 1] range (same as IXI preprocessing)
    
    Args:
        x: numpy array of image data
    
    Returns:
        Normalized array in [0, 1] range
    """
    x = x.astype(np.float32)
    if x.max() - x.min() < 1e-6:
        return np.zeros_like(x)
    return (x - x.min()) / (x.max() - x.min())


def is_valid_slice(slice_array, min_nonzero_ratio=0.12, min_mean=0.1):
    """
    Check if a slice contains meaningful information (same criteria as IXI)
    
    Args:
        slice_array: 2D numpy array
        min_nonzero_ratio: minimum ratio of non-zero pixels
        min_mean: minimum mean pixel value after normalization
    
    Returns:
        Boolean indicating if slice is valid
    """
    # Check if slice has enough non-zero pixels
    nonzero_ratio = np.count_nonzero(slice_array) / slice_array.size
    if nonzero_ratio < min_nonzero_ratio:
        return False
    
    # Normalize and check mean value
    normalized = normalize(slice_array)
    if normalized.mean() < min_mean:
        return False
    
    return True


print("✓ Preprocessing functions defined (matching IXI pipeline)!")

## 9. STEP 1: Extract and Normalize 2D Slices from T1 Volumes

This step extracts all 2D slices from 3D T1 volumes and normalizes them to [0, 1] range.
Empty or low-information slices are filtered out during extraction.

In [ ]:
# Extract 2D slices from all T1 volumes
print("="*60)
print("STEP 1: Extracting and normalizing 2D slices from T1 volumes")
print("="*60)

slice_idx = 0
corrupted_files = []
total_slices_extracted = 0
skipped_empty_slices = 0

for t1_file in tqdm(t1_files, desc="Processing T1 volumes"):
    try:
        # Load the NIfTI file
        nii_img = nib.load(t1_file)
        vol = nii_img.get_fdata()
        
        # Extract each 2D slice
        Z = vol.shape[2]  # Number of slices in z-direction
        
        for s in range(Z):
            slice_2d = vol[:, :, s]
            
            # Skip empty or low-information slices
            if not is_valid_slice(slice_2d, min_nonzero_ratio=0.12, min_mean=0.1):
                skipped_empty_slices += 1
                continue
            
            # Normalize to [0, 1]
            normalized_slice = normalize(slice_2d)
            
            # Save as .npy file
            np.save(f"{t1_raw_folder}/slice_{slice_idx:06d}.npy", normalized_slice)
            
            # Also save as PNG for quick preview
            png = (normalized_slice * 255).astype(np.uint8)
            from PIL import Image
            Image.fromarray(png).save(f"{t1_raw_folder}/slice_{slice_idx:06d}.png")
            
            slice_idx += 1
            total_slices_extracted += 1
        
    except Exception as e:
        print(f"\n❌ Error processing {os.path.basename(t1_file)}: {e}")
        corrupted_files.append(t1_file)
        continue

print("\n" + "="*60)
print("STEP 1 COMPLETE")
print("="*60)
print(f"✓ Total valid slices extracted: {total_slices_extracted}")
print(f"✓ Skipped empty slices: {skipped_empty_slices}")
print(f"✓ Corrupted files: {len(corrupted_files)}")
print(f"✓ Output location: {t1_raw_folder}")
print("="*60)

In [ ]:
# Visualize some sample extracted slices
raw_files = sorted([f for f in os.listdir(t1_raw_folder) if f.endswith('.npy')])

if len(raw_files) > 0:
    # Select 9 random samples
    sample_files = random.sample(raw_files, min(9, len(raw_files)))
    
    plt.figure(figsize=(12, 12))
    for i, f in enumerate(sample_files):
        img = np.load(os.path.join(t1_raw_folder, f))
        
        plt.subplot(3, 3, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(f, fontsize=8)
        plt.axis('off')
    
    plt.suptitle('Sample Extracted & Normalized T1 Slices', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"Total extracted slices: {len(raw_files)}")
else:
    print("No slices found. Run the extraction step first.")

## 10. STEP 2: Additional Filtering (Mean > 0.1)

Apply the same filtering criteria as IXI dataset to ensure consistency.

In [ ]:
# Filter slices based on mean pixel value (same as IXI pipeline)
print("="*60)
print("STEP 2: Filtering slices (mean > 0.1)")
print("="*60)

raw_files = sorted([f for f in os.listdir(t1_raw_folder) if f.endswith('.npy')])
kept_count = 0

for f in tqdm(raw_files, desc="Filtering slices"):
    output_file_path = os.path.join(filtered_folder, f)
    
    # Skip if already filtered
    if os.path.exists(output_file_path):
        kept_count += 1
        continue
    
    # Load and check mean value
    arr = np.load(os.path.join(t1_raw_folder, f))
    if arr.mean() > 0.1:  # Same threshold as IXI
        np.save(output_file_path, arr)
        kept_count += 1

print("\n" + "="*60)
print("STEP 2 COMPLETE")
print("="*60)
print(f"✓ Filtered slices kept: {kept_count} / {len(raw_files)}")
print(f"✓ Filtered slices removed: {len(raw_files) - kept_count}")
print(f"✓ Output location: {filtered_folder}")
print("="*60)

## 11. STEP 3: Resize to 128x128 (Standard Size)

Resize all filtered slices to 128x128 to match the IXI dataset dimensions.

In [ ]:
# Resize configuration
TARGET_SIZE = (128, 128)  # Same as IXI dataset
BATCH_SIZE = 500  # Process in batches to avoid memory issues

print("="*60)
print("STEP 3: Resizing slices to 128x128")
print("="*60)
print(f"Target size: {TARGET_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print("="*60)

In [ ]:
# Batch resize all filtered slices
filtered_files = sorted([f for f in os.listdir(filtered_folder) if f.endswith('.npy')])
total = len(filtered_files)

print(f"Total filtered slices to resize: {total}")
print("Starting batch-wise resizing...\n")

resized_count = 0
error_count = 0

for i in range(0, total, BATCH_SIZE):
    batch_files = filtered_files[i : i + BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    
    print(f"Processing batch {batch_num} ({len(batch_files)} files)...")
    
    for f in tqdm(batch_files, desc=f"Batch {batch_num}"):
        src_path = os.path.join(filtered_folder, f)
        dst_path = os.path.join(resized_folder, f)
        
        # Skip if already resized
        if os.path.exists(dst_path):
            resized_count += 1
            continue
        
        try:
            arr = np.load(src_path)
            resized = resize(arr, TARGET_SIZE, mode='reflect', anti_aliasing=True)
            np.save(dst_path, resized)
            resized_count += 1
        except Exception as e:
            print(f"❌ Error resizing {f}: {e}")
            error_count += 1
            continue

print("\n" + "="*60)
print("STEP 3 COMPLETE")
print("="*60)
print(f"✓ Successfully resized: {resized_count}")
print(f"✓ Errors: {error_count}")
print(f"✓ Output location: {resized_folder}")
print("="*60)

## 12. Verify Final Processed Files

In [ ]:
# Verify the final resized files
resized_files = sorted([f for f in os.listdir(resized_folder) if f.endswith('.npy')])

print(f"📊 Final processed files: {len(resized_files)}")

# Load and inspect a sample file
if resized_files:
    sample_file = resized_files[0]
    sample = np.load(os.path.join(resized_folder, sample_file))
    
    print(f"\n✓ Sample file: {sample_file}")
    print(f"  - Shape: {sample.shape}")
    print(f"  - Min pixel value: {sample.min():.4f}")
    print(f"  - Max pixel value: {sample.max():.4f}")
    print(f"  - Mean pixel value: {sample.mean():.4f}")
    print(f"  - Data type: {sample.dtype}")
    
    # Verify dimensions match IXI
    if sample.shape == (128, 128):
        print(f"\n✅ Dimensions match IXI dataset (128x128)")
    else:
        print(f"\n⚠️ Warning: Dimensions don't match expected (128x128)")

## 13. Visualize Final Processed Slices

In [ ]:
# Visualize 9 random final processed slices
resized_files = sorted([f for f in os.listdir(resized_folder) if f.endswith('.npy')])

if len(resized_files) > 0:
    sample_files = random.sample(resized_files, min(9, len(resized_files)))
    
    plt.figure(figsize=(12, 12))
    for i, f in enumerate(sample_files):
        img = np.load(os.path.join(resized_folder, f))
        
        plt.subplot(3, 3, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(f"{f}\n{img.shape}", fontsize=8)
        plt.axis('off')
    
    plt.suptitle('Final Processed BraTS T1 Slices (128x128)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No processed slices found.")

## 14. Pipeline Summary and Statistics

In [ ]:
print("="*70)
print(" " * 15 + "BRATS 2021 PREPROCESSING SUMMARY")
print("="*70)

print(f"\n📁 INPUT:")
print(f"   - Source folder: {brats_folder}")
print(f"   - Total patient folders: {len(patient_folders)}")
print(f"   - Total T1 files: {len(t1_files)}")

print(f"\n📊 PROCESSING PIPELINE:")
print(f"   Step 1: Extract 2D slices → {len(os.listdir(t1_raw_folder))} raw slices")
print(f"   Step 2: Filter (mean > 0.1) → {len(os.listdir(filtered_folder))} filtered slices")
print(f"   Step 3: Resize to 128x128 → {len(os.listdir(resized_folder))} final slices")

print(f"\n📂 OUTPUT FOLDERS:")
print(f"   - Raw slices: {t1_raw_folder}")
print(f"   - Filtered: {filtered_folder}")
print(f"   - Resized (FINAL): {resized_folder}")

print(f"\n✅ PIPELINE COMPLETE!")
print(f"   Total processed slices ready: {len(os.listdir(resized_folder))}")

# Calculate folder sizes
import os
def get_folder_size_mb(folder):
    total = 0
    for root, dirs, files in os.walk(folder):
        for f in files:
            fp = os.path.join(root, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)

resized_size = get_folder_size_mb(resized_folder)
print(f"   Final dataset size: {resized_size:.2f} MB")

print(f"\n🎯 NEXT STEPS:")
print(f"   1. Review the processed slices above")
print(f"   2. Create zip file for Google Drive upload (next cell)")
print(f"   3. Upload to Google Drive at: MyDrive/symAD-ECNN/data/brats2021_processed/")
print(f"   4. Use in Colab for testing/anomaly detection")

print("="*70)

## 15. Create ZIP for Google Drive Upload

**IMPORTANT**: Run this cell to create a compressed zip file of the processed data for easy upload to Google Drive.

In [ ]:
import zipfile
from datetime import datetime

# Create zip file name with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"brats2021_processed_slices_{timestamp}.zip"
zip_path = os.path.join(r"c:\Users\rifad\symAD-ECNN\data", zip_filename)

print("="*60)
print("Creating ZIP file for Google Drive upload...")
print("="*60)

# Count files to zip
resized_files = [f for f in os.listdir(resized_folder) if f.endswith('.npy')]
total_files = len(resized_files)

print(f"Files to compress: {total_files}")
print(f"Output file: {zip_filename}")
print("\nCompressing... (this may take a few minutes)")

# Create zip file
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zipf:
    for f in tqdm(resized_files, desc="Zipping files"):
        file_path = os.path.join(resized_folder, f)
        # Add file to zip with relative path
        zipf.write(file_path, arcname=f)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)

print("\n" + "="*60)
print("✅ ZIP FILE CREATED SUCCESSFULLY!")
print("="*60)
print(f"📦 File: {zip_path}")
print(f"📊 Size: {zip_size_mb:.2f} MB")
print(f"📁 Contains: {total_files} .npy files")

print(f"\n📤 UPLOAD INSTRUCTIONS:")
print(f"   1. Open Google Drive in browser")
print(f"   2. Navigate to: MyDrive/symAD-ECNN/data/")
print(f"   3. Create folder: brats2021_test")
print(f"   4. Upload: {zip_filename}")
print(f"   5. In Colab, extract with: !unzip path/to/zip -d output_folder")
print("="*60)

## 16. Colab Upload & Extraction Code (Copy-Paste to Colab)

Copy the code below and run it in Google Colab after uploading the zip file.

In [ ]:
# ========================================
# COPY THIS TO GOOGLE COLAB
# ========================================

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile

# Define paths (adjust if needed)
BASE = "/content/drive/MyDrive/symAD-ECNN"
zip_file = f"{BASE}/data/brats2021_processed_slices_XXXXXXXX_XXXXXX.zip"  # Update with your zip filename
output_folder = f"{BASE}/data/brats2021_test"

# Create output folder
os.makedirs(output_folder, exist_ok=True)

# Extract zip file
print(f"Extracting {zip_file}...")
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(output_folder)

# Verify
import numpy as np
files = sorted([f for f in os.listdir(output_folder) if f.endswith('.npy')])
print(f"\n✅ Extraction complete!")
print(f"Total files: {len(files)}")

# Load and verify a sample
if files:
    sample = np.load(os.path.join(output_folder, files[0]))
    print(f"\nSample file: {files[0]}")
    print(f"Shape: {sample.shape}")
    print(f"Range: [{sample.min():.4f}, {sample.max():.4f}]")

# ========================================
# END OF COLAB CODE
# ========================================